# Tutorial for gathering construction-site imagery


This notebook walks through the two-step pipeline from [A Framework for Semi-automatic Collection of Temporal Satellite Imagery for Analysis of Dynamic Regions](https://openaccess.thecvf.com/content/ICCV2021W/LUAI/papers/Motlagh_A_Framework_for_Semi-Automatic_Collection_of_Temporal_Satellite_Imagery_for_ICCVW_2021_paper.pdf) (ICCVW 2021): **extract** OpenStreetMap construction sites, then **gather** Sentinel-2 or Planet chips over each site's construction window.

This is the 2026 personal-fork modernization (Python 3.10+, ohsome extract by default, Planet PSScene Orders API, Sentinel-2 via Planetary Computer STAC or the Copernicus Data Space Process API). Original authors: Nicholas Kashani Motlagh, Aswathnarayan Radhakrishnan, Jim Davis (Ohio State University), and Roman Ilin (AFRL/RYAP). Point of contact for this fork is Nick ([@nmotlagh](https://github.com/nmotlagh)).

There is no GUI on purpose: ohsome queries and osmium jobs can be long, and imagery orders consume quota.


## Prerequisites

Complete the install and credentials steps in [README.md](./README.md) first:

1. `pip install -e ".[dev,notebook]"`.
2. `cp .env.example .env` and fill in keys if you will use Planet or the CDSE Process API. **Do not paste secrets into this notebook.**
3. Optional: system package `osmium-tool` only if you use `--backend osmium`.

Planet: set `PLANET_API_KEY` **or** run `planet auth login`. Sentinel-2 STAC (`source = "stac"`) needs no key. Process API (`source = "s"`): create a CDSE OAuth client at the [Copernicus Data Space dashboard](https://shapps.dataspace.copernicus.eu/dashboard/#/account/settings) and set `SH_CLIENT_ID` / `SH_CLIENT_SECRET`. Old Sentinel Hub instance IDs will not work. Nick may need new keys.


### Acquiring history files


Default extract uses the ohsome API and does **not** need a history dump. Only `--backend osmium` needs a Geofabrik **full-history** `.osh.pbf`. Create a free [OpenStreetMap account](https://www.openstreetmap.org/user/new), then:

* Open the Ohio page: [download.geofabrik.de/north-america/us/ohio.html](https://download.geofabrik.de/north-america/us/ohio.html).
* Under **Other Formats and Auxiliary Files**, follow **internal server** ([osm-internal.download.geofabrik.de](https://osm-internal.download.geofabrik.de/)).
* Log in with your OSM account and download the previously crossed-out `ohio-internal.osh.pbf`.

Other regions: [download.geofabrik.de](https://download.geofabrik.de). Keep `*.osh.pbf` out of git.


### Creating poly files


`extract_sites` uses an [Osmosis `.poly`](https://wiki.openstreetmap.org/wiki/Osmosis/Polygon_Filter_File_Format) as the search region. ohsome uses the poly bbox; osmium also clips the history file to it.

`poly/campus.poly` is an **approximate bounding box** around the Ohio State University Columbus campus, for this demo only. It is not a cadastral campus boundary.


## Step 1: site extraction

Create working directories, then build a GeoDataFrame of construction sites. Default is ohsome (no `.osh.pbf`). CLI equivalent: `cssic setup` then `cssic extract ...`.


### Setup

```
temp/
    snapshots/
output/
    collection/
```


In [ ]:
from workspace import setup_directory

setup_directory()


### Reset extract outputs

This deletes `temp/` snapshots, `output/collection/`, and per-site folders created during extraction. CLI: `cssic reset-extract`.


In [ ]:
import reset_extract

# reset_extract.reset()


### Obtaining GeoDataFrames

`extract_sites` records sites that were under construction between `--start` and `--end` and writes `output/collection/collection.gpkg` (plus GeoJSON and Shapefile copies). Default `--backend ohsome` queries HeiGIT; no `.osh.pbf` required. Sites that began or ended outside the window but were active inside it still appear with their true dates unless `restrict-window` is set.

Each row has `chain_id`, `start`, `end`, `constr_tag`, `prev_tag`, `final_tag`, and a bounding-box polygon. ohsome sets `prev_tag` / `final_tag` to `N/A` (it does not reconstruct WayChain ID-rewiring).


In [ ]:
import extract_sites


Parameters:

* `start`: on or after **2015-06-22** (`YYYY-MM-DD`).
* `end`: at least **10 days before today**.
* `poly`: Osmosis poly path. Demo uses `poly/campus.poly`.
* `backend`: `"ohsome"` (default) or `"osmium"`.
* `region`: path to the `.osh.pbf` history file (**osmium only**; leave empty for ohsome). Do not commit the file.
* `keep-temp`: keep daily osmium snapshots.
* `restrict-window`: do not walk before start / after end for true dates.
* `save-wip`: also write `output/collection/in_progress.*`.


In [ ]:
# Search window (must end at least 10 days before today)
start_date = "2018-04-15"
end_date = "2020-07-01"
poly_file = "poly/campus.poly"
backend = "ohsome"  # or "osmium"
# Local path to ohio-internal.osh.pbf (osmium only; not committed)
region_file = ""
keep_temp_files = False
restrict_window = False
save_wip = False


In [ ]:
params = {
    "start": start_date,
    "end": end_date,
    "poly": poly_file,
    "backend": backend,
    "region": region_file,
    "keep-temp": keep_temp_files,
    "restrict-window": restrict_window,
    "save-wip": save_wip,
}
extract_sites.set_params(params)


`locate_construction()` uses ohsome by default (a network call; no osmium). `--backend osmium` shells out to osmium and can take a long time. It returns in-progress and completed GeoDataFrames and writes the completed collection under `output/collection/`.


In [ ]:
in_progress_gdf, collection_gdf = extract_sites.locate_construction()


In [ ]:
collection_gdf


In [ ]:
collection_gdf.plot()


`create_dataset` writes `output/{chain_id}/info.txt` for each completed site (`start,end,prev_tag,final_tag,minx,miny,maxx,maxy`).


In [ ]:
extract_sites.create_dataset(collection_gdf)


Distribution of estimated final OSM tags:


In [ ]:
import matplotlib.pyplot as plt

histogram = collection_gdf["final_tag"].value_counts()
plt.xticks(rotation=90)
plt.bar(histogram.index, histogram.values)


## Step 2: image gathering

Download RGB and/or NIR chips over each site's construction window. CLI equivalent: `cssic gather -s stac --rgb --nir`, `cssic gather -s p --rgb --nir`, or `cssic gather -s s --rgb --nir`.

Credentials are loaded from `.env` (see `.env.example`). Do not hard-code keys here. STAC needs none.


In [ ]:
from pathlib import Path
import os

from dotenv import load_dotenv
import gather_images

load_dotenv(Path(".env"))
print("PLANET_API_KEY set:", bool(os.getenv("PLANET_API_KEY")))
print("SH_CLIENT_ID set:", bool(os.getenv("SH_CLIENT_ID")))
print("SENTINEL_PROVIDER:", os.getenv("SENTINEL_PROVIDER", "cdse"))


### Reset downloaded imagery

Removes chips under `output/{chain_id}/images/` but leaves the collection table. CLI: `cssic reset-images`.


In [ ]:
import reset_images

# reset_images.reset()


### Planet vs Sentinel-2

**Sentinel-2 STAC** (`source = "stac"`) searches Planetary Computer and clips chips. No CDSE key. Sites before **2015-06-23** are skipped.

**Planet (PSScene Orders API)** is two steps: place orders, then download when they succeed. Sites that started before **2017-02-19** are skipped. Coverage is thin in 2017. Set `email=True` to get Planet's ready-to-download mail, or re-run later with `download-planet=True`. Auth is `PLANET_API_KEY` or `planet auth login`.

**Sentinel-2 Process API** (`source = "s"`) is one request per sample via Copernicus Data Space (`SENTINEL_PROVIDER=cdse`). Sites before **2015-06-23** are skipped. Create an OAuth client on the [CDSE dashboard](https://shapps.dataspace.copernicus.eu/dashboard/#/account/settings). Legacy Sentinel Hub instance IDs / configuration layers are not used anymore.


Gather flags:

* `source`: `"stac"`, `"p"` / `"planet"`, or `"s"` / `"sentinel"`.
* `rgb` / `nir`: at least one required.
* `num-images`: `>= 3` evenly spaced samples (including start and end), or `-1` for every available date.
* `padding`: area scale of the bounding box (center-invariant; `1` = exact box).
* `verbose` / `email`: logs, and Planet email notifications.
* `download-planet`: download previously created Planet orders (`cssic gather ... --download`).
* `api`: optional Planet key override; leave `None` to use the environment.
* `sh-client-id` / `sh-client-secret` / `sentinel-provider`: optional CDSE overrides; default provider is `cdse`.


In [ ]:
rgb = True
nir = True
source = "stac"  # "stac", "p" / "planet", or "s" / "sentinel"
num_images = 3
padding = 1
verbose = True
email = False
download_planet = False

# Leave these None to use .env / `planet auth login`. Do not paste secrets here.
api = None
sh_client_id = None
sh_client_secret = None
sentinel_provider = "cdse"

params = {
    "rgb": rgb,
    "nir": nir,
    "source": source,
    "num-images": num_images,
    "padding": padding,
    "verbose": verbose,
    "email": email,
    "download-planet": download_planet,
    "api": api,
    "sh-client-id": sh_client_id,
    "sh-client-secret": sh_client_secret,
    "sentinel-provider": sentinel_provider,
}
gather_images.set_params(params)


`gather_images.setup()` creates per-site image folders and loads the newest collection (`collection.gpkg`, then `.geojson`, then `.shp`).


In [ ]:
collection_gdf = gather_images.setup()


Place Planet orders or pull Sentinel-2 chips. Planet and CDSE Process API use quota; STAC does not. Planet prints a reminder to re-run with `--download` once orders succeed.


In [ ]:
gather_images.gather_from_source(collection_gdf)


If `source` is Planet, wait until orders succeed, then download. CLI: `cssic gather -s p --rgb --nir --download`.


In [ ]:
params["download-planet"] = True
gather_images.set_params(params)
gather_images.gather_from_source(collection_gdf)


Chips are saved at `output/{chain_id}/images/{planet|sentinel}/{rgb|nir}/{date_acquired}.png`.

Planet RGB may include `{date}_mask.png`. Planet NIR may include `{date}_udm.tif`. With `num-images = -1`, missing days get `{date}_empty.png`.
